In [ ]:
## ติดตั้ง thai llm
## ติดตั้ง langchain
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q langchain-google-genai
!pip install -q langchain-huggingface
!pip install -q markdown2
!pip install -q fpdf pypdf
!pip install -q pinecone ## เชื่อมต่อ vector database
!pip install -q langchain-pinecone
!pip install -q FlagEmbedding ## เพิ่มติดตั้ง FlagEmbedding ที่นี่
!pip install -q transformers==4.29.2 ## ดาวน์เกรด transformers เพื่อแก้ปัญหา ImportError กับ FlagEmbedding

## ติดตั้งสำหรับการแปรงไฟล์จาก text เป็น pdf
## เพื่อแปลงเป็นไฟล์ pdf หาไลบรารีอันนี้กินเวลาชิปเป้ง
## ลำดับ playwright ผิด ต้อง pip install ก่อน แล้วค่อย install chromium
!pip install -q playwright
!playwright install-deps chromium
!playwright install chromium

## ติดตั้งฟอนต์ไทย
!wget -q https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf -O Sarabun-Regular.ttf

In [ ]:
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec
from google.colab import userdata   ## เพื่อจัดการ api key
from google.colab import drive
from pathlib import Path            ## เพื่อจัดการ path ไฟล์แบบ
import markdown as md               ## เพื่อจัดการ markdown
import pandas as pd
import numpy as np
import torch
import html                         ## เพื่อแปลง html entity
import csv                          ## เพื่ออ่าน/เขียนไฟล์ CSV
import re                           ## เพื่อทำ pattern matching ข้อความ
import os


## เพื่อการทำงานของ api
import requests
import time
llm_api_key = userdata.get('Typhoon')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
drive.mount('/content/drive')

project_dir     =  Path('/content/drive/MyDrive/super-ai-engineer-season-6-rag-2569')
data_path       = project_dir / 'data'
knowledge_path  = data_path / 'knowledge_base'
policy_path     = knowledge_path  / 'policies'                      # folder ที่เก็บนโยบายทั้งหมด
product_path    = knowledge_path / 'products'                       # folder ที่เก็บสินค้าทั้งหมด
store_path      = knowledge_path / 'store_info'                     # folder ที่เก็บที่อยู่ทั้งหมด
question_path   = data_path / 'questions.csv'
output_path     = data_path / 'sample_submission.csv'

In [ ]:
## การตั้งค่า API จากอาจารย์
def ask_llm(messages, model = "kbtg", max_retries = 5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": llm_api_key}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2048,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None

In [ ]:
## รวมเอกสารทั้งหมดเข้าด้วยกัน
# แก้ไขโดยใช้ Path() เพื่อให้สามารถใช้ .glob() ได้
knowledge_base = Path(knowledge_path)
document = []

## glob  จะทำการค้นหาไฟล์ .md เฉพาะโฟลเดอร์ที่ละบุเท่านั้น
## rglob จะทำการค้นหาไฟล์ .md ทั้งโฟลเดอร์ใหญ่และโฟลเดอร์ย่อย
if knowledge_base.exists():
    for fp in sorted(knowledge_base.rglob('*.md')):
        doc = fp.read_text(encoding = 'utf-8')
        document.append({'path' : str(fp.relative_to(knowledge_base)), 'text' : doc})

In [ ]:
## ทำความสะอาดตัวอักษรจาก markdown ก่อน
def clean_markdown(text):
    text = re.sub(r'#+\s+', '', text)
    text = re.sub(r'\*\*', '', text)
    text = re.sub(r'---', '', text)
    text = re.sub(r'—', '', text)
    text = re.sub(r'\|\|', '', text)
    text = re.sub(r'\|[\s\-\:\|]+\|', '', text)
    text = re.sub(r'\|', '', text)
    return text.strip()

In [ ]:
text_document = document[0]['text']
clean_text = clean_markdown(text_document)

In [ ]:
## ทำความสะอาดทีละไฟล์
all_cleaned_text = []

for doc_item in document:
    cleaned = clean_markdown(doc_item['text'])
    all_cleaned_text.append(cleaned)

In [ ]:
## รวมไฟล์ทุกอันเข้าด้วยกัน
final_combined_text = '\n\n---\n\n'.join(all_cleaned_text)

In [ ]:
## ลองเซฟไฟล์ออกมาเป็นนามสกุล .txt
txt_output_path = data_path / 'all_clean_documents.txt'

with open(txt_output_path, 'w', encoding = 'utf-8') as f:
    f.write(final_combined_text)

print(f"รวมไฟล์ทั้งหมด {len(document)} ไฟล์ และบันทึกเรียบร้อยแล้วที่: {txt_output_path}")
print(f"ความยาวตัวอักษรรวม: {len(final_combined_text)} ตัวอักษร")

In [ ]:
from playwright.async_api import async_playwright
import asyncio

async def create_final_pdf():
## 1. อ่านไฟล์ txt ก่อนทำ pdf
    with open('/content/drive/MyDrive/super-ai-engineer-season-6-rag-2569/data/all_clean_documents.txt', 'r', encoding='utf-8') as f:
        content = f.read()

## 2. แปลงเนื้อหาเป็น HTML (ใช้ฟอนต์สารบรรณจาก Google เพื่อความสวยงาม)
    html_content = content.replace('\n', '<br>')
    full_html = f"""
    <html>
    <head>
        <meta charset='utf-8'>
        <link href='https://fonts.googleapis.com/css2?family=Sarabun&display=swap' rel='stylesheet'>
        <style>
            body {{ font-family: 'Sarabun', sans-serif; padding: 40px; line-height: 1.6; font-size: 11pt; }}
        </style>
    </head>
    <body>{html_content}</body>
    </html>
    """

## 3. สั่งเซฟเป็น PDF
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.set_content(full_html)
        await page.pdf(path='fahMai_fixed_final.pdf', format='A4', margin={'top':'12.7mm', 'bottom':'12.7mm', 'left':'12.7mm', 'right':'12.7mm'})
        await browser.close()
        print('เสร็จแล้วครับ!')

await create_final_pdf()

In [ ]:
## สกัดเนื้อหาสาระจากไฟล์ pdf

txt = Path('/content/fahMai_fixed_final.pdf')

def load_pdf_files(data):
    loader = DirectoryLoader(
        data, # DirectoryLoader requires a directory path string
        glob = '*.pdf',
        loader_cls = PyPDFLoader,
    )

    document_x = loader.load()
    return document_x

In [ ]:
extract_data = load_pdf_files('/content')

In [ ]:
extract_data

In [ ]:
## จำนวนหน้าทั้งหมดของไฟล์ pdf อันนี้
len(extract_data)

245

In [ ]:
## ลดจำนวน metadata เพื่อช่วย llm ประหยัดความจำ ลดความสับสน
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get('source')
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {'source': src}
            )
        )
    return minimal_docs

In [ ]:
## ลดจำนวน metadata เพื่อช่วย llm ประหยัดความจำ ลดความสับสน
minimal_docs = filter_to_minimal_docs(extract_data)

In [ ]:
minimal_docs

[Document(metadata={'source': '/content/fahMai_fixed_final.pdf'}, page_content='นโยบายการยกเลิกคําสั\x00งซื\x00อ  ร้านฟ\x00าใหม่\nวันที\x00อัปเดต : 1 มีนาคม  2569\n1. ภาพรวมนโยบาย\nฟ\x00าใหม่เข้าใจว่าบางครั\x00งลูกค้าอาจต้องการยกเลิกคําสั\x00งซื\x00อด้วยเหตุผลต่างๆ  นโยบายนี\x00อธิบายสิทธิ\x00และขั\x00น\nตอนการยกเลิกคําสั\x00งซื\x00อตามสถานะของคําสั\x00งซื\x00อในขณะนั\x00น  ความสามารถในการยกเลิกขึ\x00นอยู่กับสถานะ\nคําสั\x00งซื\x00อเป\x00นหลัก\n2. การยกเลิกตามสถานะคําสั\x00งซื\x00อ\n2.1 สถานะ  " รอชําระเงิน " (Pending Payment)\nยกเลิกได้ทันที\nคําสั\x00งซื\x00อที\x00อยู่ในสถานะรอชําระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย  ผ่านแอปพลิเคชัน  FahMai\nหรือเว็บไซต์  www.fahmai.th\nAuto-cancel: หากไม่มีการชําระเงินภายใน  24 ชั\x00วโมง  นับจากเวลาที\x00สร้างคําสั\x00งซื\x00อ  ระบบจะยกเลิกคําสั\x00ง\nซื\x00อโดยอัตโนมัติ  และสินค้าจะกลับไปยังคลังสินค้า\n> ไม่มีการหักค่าธรรมเนียมใดๆ  สําหรับการยกเลิกในสถานะนี\x00\n2.2 สถานะ  " ชําระเงินแล้ว  กําลังเตรียมจัดส่ง " (Processing / Preparing to Sh

In [ ]:
## split the document into smaller chunks ปรับเหลือน้อย เพราะเน้นบริบท แล้วก็ปรับเพิ่ม เพราะ chunk มาไม่ครบบริบท
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f'number of chunks: {len(texts_chunk)}')

number of chunks: 523


In [ ]:
texts_chunk

In [ ]:
## necessary part : embedding
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
# !pip install "numpy<2.0.0"

# # หลังจากรันบรรทัดบนเสร็จ แนะนำให้เมนู Runtime > Restart session ก่อนรันโค้ดด้านล่างครับ

# import torch
# from langchain_huggingface import HuggingFaceEmbeddings

# model_name = "BAAI/bge-m3"
# model_kwargs = {"device": "cuda" if torch.cuda.is_available() else "cpu"}
# encode_kwargs = {"normalize_embeddings": True}

# embedding = HuggingFaceEmbeddings(
#     model_name = model_name,
#     model_kwargs = model_kwargs,
#     encode_kwargs = encode_kwargs,
# )

# embedding_query = HuggingFaceEmbeddings(
#     model_name = model_name,
#     model_kwargs = model_kwargs,
#     encode_kwargs = encode_kwargs,
# )

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# ตอน upsert
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview",
    google_api_key=userdata.get('Gemini'),
    task_type="retrieval_document"
)

# ตอน retrieve
embedding_query = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview",
    google_api_key=userdata.get('Gemini'),
    task_type="retrieval_query"
)

In [ ]:
embedding

In [ ]:
# def download_embeddings():
#     ## ลองอันนี้แล้ว sentence-transformers/all-MiniLM-L6-v2 อ่อนภาษาไทยมาก
#     model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
#     embeddings = HuggingFaceEmbeddings(
#         model_name = model_name,
#         model_kwargs = {'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
#     )
#     return embeddings

# embedding = download_embeddings()

In [ ]:
## สำรวจมิติของการ embedding จาก model ฟรี
vector = embedding_query.embed_query('สวัสดีโลก')

print('vector length: ', len(vector))

vector length:  1024


In [ ]:
## เชื่อมต่อ api ของ pinecone และ open
PINECONE_API_KEY = userdata.get('Pinecone')
OPENAI_API_KEY = userdata.get('OpenAI')

In [ ]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key = pinecone_api_key)

In [ ]:
## ตั้งค่าสำหรับทำ vector database ที่ pinecone
index_name = 'open-fahmai-chatbot'

if not pc.has_index(index_name): ## ตรวจชื่อห้องกรณีอาจจะมีซ้ำ
    pc.create_index(
        name = index_name,
        dimension = len(vector),    ## ตั้งค่ามิติของ embedding
        metric = 'cosine',          ## similarity metrics
        spec = ServerlessSpec(
            cloud = 'aws',
            region = 'us-east-1'
        )
    )

index = pc.Index(index_name)


In [ ]:
from langchain_pinecone import PineconeVectorStore

# ตั้งค่า Environment Variable เพื่อความมั่นใจ
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

docsearch = PineconeVectorStore.from_documents(
    documents = texts_chunk,
    embedding = embedding,
    index_name = index_name
)

In [ ]:
## เชื่อมต่อกับความจำเดิมที่เคยบันทึก llm พร้อมใช้งานทันที

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding = embedding_query
)


In [ ]:
# ## เชื่อมต่อกับความจำเดิมที่เคยบันทึก llm พร้อมใช้งานทันที

# from langchain_pinecone import PineconeVectorStore

# docsearch = PineconeVectorStore.from_existing_index(
#     index_name = index_name,
#     embedding = embedding
# )


In [ ]:
# from langchain_community.retrievers import BM25Retriever
# from langchain_classic.retrievers import EnsembleRetriever
# from sentence_transformers import CrossEncoder  # ← เปลี่ยนตรงนี้

# ## BM25 retriever (keyword-based)
# bm25_retriever = BM25Retriever.from_documents(texts_chunk)
# bm25_retriever.k = 10

# ## Vector retriever (semantic)
# vector_retriever = docsearch.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": 10}
# )

# ## Hybrid: รวม BM25 + Vector ด้วย EnsembleRetriever
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, vector_retriever],
#     weights=[0.5, 0.5]
# )

# ## Reranker: BGE-Reranker-V2-M3 คัด top-k จริงๆ
# reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")  # ← เปลี่ยนตรงนี้

# def retrieve_and_rerank(query, top_k=5):
#     candidates = ensemble_retriever.invoke(query)
#     if not candidates:
#         return []
#     pairs = [[query, doc.page_content] for doc in candidates]
#     scores = reranker.predict(pairs)  # ← เปลี่ยนตรงนี้
#     ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
#     return [doc for _, doc in ranked[:top_k]]

In [ ]:
# from pinecone.db_data.dataclasses import search_rerank
# retriever = docsearch.as_retriever(search_type = 'similarity', search_kwargs = {'k': 9})

In [ ]:
retrieved_docs = retriever('Watch S3 Ultra กันน้ำได้กี่ ATM ครับ')

retrieved_docs

[Document(metadata={'source': '/content/fahMai_fixed_final.pdf', 'text': 'A: ว่ายน\x00าได้สบายเลยค่ะ  กันน\x00า  5 ATM (50 เมตร ) รองรับว่ายน\x00าในสระและทะเล  แต่ไม่มีโหมด  Dive\nMode สําหรับการดําน\x00าลึก  ถ้าต้องการ  Dive Mode แนะนํา  Watch S3 Ultra (WK-SW-001) ที\x00กันน\x00า  10\nATM และมีโหมดเฉพาะสําหรับการดําน\x00า\nQ: NFC Pay ใช้ชําระเงินได้ที\x00ไหนบ้างครับ ?\nA: ใช้ได้ที\x00เครื\x00อง  PoS ที\x00รองรับการชําระเงินแบบ  contactless ผ่านระบบ  FahMai Pay ครับ  ต้องเชื\x00อม\nโยงบัตรหรือบัญชีผ่านแอป  WongKhoJon Health ก่อนใช้งาน\nQ: ข้อมูล  ECG แชร์ให้แพทย์ได้ไหมครับ ?\nA: ได้ครับ  ข้อมูล  ECG จะบันทึกใน  PDF รูปแบบมาตรฐานผ่านแอป  WongKhoJon Health สามารถส่ง\nออกเป\x00นไฟล์  PDF หรือแชร์ผ่านอีเมลให้แพทย์ได้โดยตรง  ข้อมูลนี\x00เป\x00นข้อมูลเบื\x00องต้นเพื\x00อประกอบการ\nพิจารณาของแพทย์  ไม่ใช่การวินิจฉัยโรค\n---\nวงโคจร  Watch S3 (WongKhoJon Watch S3)\nรหัสสินค้า : WK-SW-003\nแบรนด์ : วงโคจร  (WongKhoJon) แบรนด์ในเครือฟ\x00าใหม่\nหมวดหมู่ : สมาร์ทวอทช์'}, page_content='A: ว่ายน\x0

In [ ]:
## เฉพาะพาร์ทนี้เริ่มตันหัวล่ะ มาราธอนทั้งวันน
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any


class ThaiLLM(LLM):
    model: str = "kbtg"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs: Any) -> str:
        messages = [{"role": "user", "content": prompt}]
        result = ask_llm(messages, model=self.model)
        return result or ""

    @property
    def _llm_type(self) -> str:
        return "thai_llm"

In [ ]:
from langchain_openai import ChatOpenAI
## chatModel = ChatOpenAI(model = 'gpt-4o')

chatModel = ThaiLLM(model="kbtgt")

In [ ]:
# ติดตั้งแบบระบุเวอร์ชันที่เสถียรและทำงานร่วมกันได้ดี
!pip install -U langchain-classic


# จากเดิม langchain.chainsเปลี่ยนเป็น langchain_classic.chains เพราะ error ไม่มีเวลาอ่าน doc
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "คุณเป็นผู้ช่วยตอบคำถามของร้านฟ้าใหม่ ร้านอิเล็กทรอนิกส์\n"
    "อ่านข้อมูลอ้างอิงทุก chunk ให้ครบและละเอียดก่อนตอบ\n"
    "ข้อมูลที่ต้องการอาจอยู่ใน chunk ใดก็ได้ ไม่จำเป็นต้องอยู่ chunk แรก\n\n"
    "กฎเหล็ก:\n"
    "1. ถ้าพบข้อมูลที่เกี่ยวข้องใน chunk ใดก็ตาม ให้ตอบตัวเลขของตัวเลือกที่ตรงที่สุดเสมอ\n"
    "2. ตอบ 9 เฉพาะเมื่ออ่านครบทุก chunk แล้วไม่มีข้อมูลที่เกี่ยวข้องจริงๆ เท่านั้น\n"
    "3. ตอบ 10 เฉพาะเมื่อคำถามไม่เกี่ยวกับร้านฟ้าใหม่หรือสินค้าในร้านเลย\n"
    "4. ห้ามตอบ 9 ถ้ายังมีข้อมูลใน chunk ที่เกี่ยวข้องกับคำถามแม้แต่น้อย\n"
    "5. ตอบเฉพาะตัวเลข 1-10 เท่านั้น ห้ามตอบอย่างอื่นเด็ดขาด"
    "\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
## ไม่ได้ใช้งาน pipeline ของ langchain แล้วว
# question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
# rag_chain = create_retrieval_chain(retrieve, question_answer_chain)

In [ ]:
# response = rag_chain.invoke({'input': 'หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ'})
# print(response['answer'])

In [ ]:
# ทดสอบ pipeline ก่อนรัน 100 ข้อจริง
questions_df = pd.read_csv(question_path)
test_row = questions_df.iloc[0]
retrieved = retrieve_and_rerank(test_row['question'], top_k=5)
context = "\n\n".join([doc.page_content for doc in retrieved])

choices_text = "\n".join([
    f"ตัวเลือกที่ {i}: {test_row[f'choice_{i}']}"
    for i in range(1, 11)
])

messages = [
    {
        "role": "system",
        "content": (
            "คุณเป็นผู้ช่วยตอบคำถามของร้านฟ้าใหม่ ร้านอิเล็กทรอนิกส์\n"
            "อ่านข้อมูลอ้างอิงทุก chunk ให้ครบและละเอียดก่อนตอบ\n"
            "ข้อมูลที่ต้องการอาจอยู่ใน chunk ใดก็ได้ ไม่จำเป็นต้องอยู่ chunk แรก\n\n"
            "กฎเหล็ก:\n"
            "1. ถ้าพบข้อมูลที่เกี่ยวข้องใน chunk ใดก็ตาม ให้ตอบตัวเลขของตัวเลือกที่ตรงที่สุดเสมอ\n"
            "2. ตอบ 9 เฉพาะเมื่ออ่านครบทุก chunk แล้วไม่มีข้อมูลที่เกี่ยวข้องจริงๆ เท่านั้น\n"
            "3. ตอบ 10 เฉพาะเมื่อคำถามไม่เกี่ยวกับร้านฟ้าใหม่หรือสินค้าในร้านเลย\n"
            "4. ห้ามตอบ 9 ถ้ายังมีข้อมูลใน chunk ที่เกี่ยวข้องกับคำถามแม้แต่น้อย\n"
            "5. ตอบเฉพาะตัวเลข 1-10 เท่านั้น ห้ามตอบอย่างอื่นเด็ดขาด"
        )
    },
    {
        "role": "user",
        "content": (
            f"ข้อมูลอ้างอิง:\n{context}\n\n"
            f"คำถาม: {test_row['question']}\n\n"
            f"{choices_text}\n\n"
            "ตอบเฉพาะตัวเลข 1-10:"
        )
    }
]

raw = ask_llm(messages, model="kbtg")
print("คำถาม:", test_row['question'])
print("raw output:", repr(raw))  # ดูว่า LLM ตอบรูปแบบไหน

In [ ]:
# # ทดสอบ pipeline ก่อนรัน 100 ข้อจริง
# questions_df = pd.read_csv(question_path)
# test_row = questions_df.iloc[0]
# retrieved = retriever.invoke(test_row['question'])
# context = "\n\n".join([doc.page_content for doc in retrieved])

# choices_text = "\n".join([
#     f"ตัวเลือกที่ {i}: {test_row[f'choice_{i}']}"
#     for i in range(1, 11)
# ])

# messages = [
#     {
#         "role": "system",
#         "content": (
#             "คุณเป็นผู้ช่วยตอบคำถามของร้านฟ้าใหม่ ร้านอิเล็กทรอนิกส์\n"
#             "อ่านข้อมูลอ้างอิงทุก chunk ให้ครบและละเอียดก่อนตอบ\n"
#             "ข้อมูลที่ต้องการอาจอยู่ใน chunk ใดก็ได้ ไม่จำเป็นต้องอยู่ chunk แรก\n\n"
#             "กฎเหล็ก:\n"
#             "1. ถ้าพบข้อมูลที่เกี่ยวข้องใน chunk ใดก็ตาม ให้ตอบตัวเลขของตัวเลือกที่ตรงที่สุดเสมอ\n"
#             "2. ตอบ 9 เฉพาะเมื่ออ่านครบทุก chunk แล้วไม่มีข้อมูลที่เกี่ยวข้องจริงๆ เท่านั้น\n"
#             "3. ตอบ 10 เฉพาะเมื่อคำถามไม่เกี่ยวกับร้านฟ้าใหม่หรือสินค้าในร้านเลย\n"
#             "4. ห้ามตอบ 9 ถ้ายังมีข้อมูลใน chunk ที่เกี่ยวข้องกับคำถามแม้แต่น้อย\n"
#             "5. ตอบเฉพาะตัวเลข 1-10 เท่านั้น ห้ามตอบอย่างอื่นเด็ดขาด"
#         )
#     },
#     {
#         "role": "user",
#         "content": (
#             f"ข้อมูลอ้างอิง:\n{context}\n\n"
#             f"คำถาม: {test_row['question']}\n\n"
#             f"{choices_text}\n\n"
#             "ตอบเฉพาะตัวเลข 1-10:"
#         )
#     }
# ]

# raw = ask_llm(messages, model="typhoon")
# print("คำถาม:", test_row['question'])
# print("raw output:", repr(raw))  # ดูว่า LLM ตอบรูปแบบไหน

In [ ]:
retrieved = retrieve_and_rerank('Watch S3 Ultra กันน้ำได้กี่ ATM ครับ')
for i, doc in enumerate(retrieved):
    print(f"--- chunk {i+1} ---")
    print(doc.page_content)
    print()


# แก้ไข: ใช้ test_row แทน row เพื่อให้สามารถรันคำสั่งด้านล่างได้โดยไม่ติด error
retrieved = retrieve_and_rerank(test_row['question'])
context = "\n\n".join([doc.page_content for doc in retrieved])

In [ ]:
def rag_answer(row):
    retrieved = retrieve_and_rerank(row['question'], top_k=5)
    context = "\n\n".join([doc.page_content for doc in retrieved]) ## ไม่ลองแบบ reverse แล้ว

    choices_text = "\n".join([
        f"ตัวเลือกที่ {i}: {row[f'choice_{i}']}"
        for i in range(1, 11)
    ])

    messages = [
        {
            "role": "system",
            "content": (
                "คุณเป็นผู้ช่วยตอบคำถามของร้านฟ้าใหม่ ร้านอิเล็กทรอนิกส์\n"
                "อ่านข้อมูลอ้างอิงทุก chunk ให้ครบและละเอียดก่อนตอบ\n"
                "ข้อมูลที่ต้องการอาจอยู่ใน chunk ใดก็ได้ ไม่จำเป็นต้องอยู่ chunk แรก\n\n"
                "กฎเหล็ก:\n"
                "1. ถ้าพบข้อมูลที่เกี่ยวข้องใน chunk ใดก็ตาม ให้ตอบตัวเลขของตัวเลือกที่ตรงที่สุดเสมอ\n"
                "2. ตอบ 9 เฉพาะเมื่ออ่านครบทุก chunk แล้วไม่มีข้อมูลที่เกี่ยวข้องจริงๆ เท่านั้น\n"
                "3. ตอบ 10 เฉพาะเมื่อคำถามไม่เกี่ยวกับร้านฟ้าใหม่หรือสินค้าในร้านเลย\n"
                "4. ห้ามตอบ 9 ถ้ายังมีข้อมูลใน chunk ที่เกี่ยวข้องกับคำถามแม้แต่น้อย\n"
                "5. ตอบเฉพาะตัวเลข 1-10 เท่านั้น ห้ามตอบอย่างอื่นเด็ดขาด"
            )
        },
        {
            "role": "user",
            "content": (
                f"ข้อมูลอ้างอิง:\n{context}\n\n"
                f"คำถาม: {row['question']}\n\n"
                f"{choices_text}\n\n"
                "ตอบเฉพาะตัวเลข 1-10:"
            )
        }
    ]

    raw = ask_llm(messages, model="typhoon")

    if raw:
        match = re.search(r'\b([1-9]|10)\b', raw)
        if match:
            return int(match.group(1))
    return 9  # default

# วนทุกข้อ
answers = []
for _, row in questions_df.iterrows():
    ans = rag_answer(row)
    print(f"Q{row['id']}: {ans}")
    answers.append({'id': int(row['id']), 'answer': ans})
    pd.DataFrame(answers).to_csv(output_path, index=False)  # save ทุกข้อ

print(f"\nเสร็จแล้ว! บันทึกที่: {output_path}")

Q1: 5
Q2: 7
Q3: 2
Q4: 6
Q5: 6
Q6: 8
Q7: 1
Q8: 4
Q9: 1
Q10: 2
Q11: 1
Q12: 1
Q13: 1
Q14: 1
Q15: 7
Q16: 1
Q17: 8
Q18: 5
Q19: 2
Q20: 1
Q21: 3
Q22: 1
Q23: 1
Q24: 3
Q25: 5
Q26: 6
Q27: 2
Q28: 7
Q29: 4
Q30: 1
Q31: 9
Q32: 1
Q33: 8
Q34: 10
Q35: 3
Q36: 2
Q37: 8
Q38: 6
Q39: 4
Q40: 8
Q41: 7
Q42: 2
Q43: 4
Q44: 1
Q45: 1
Q46: 1
Q47: 1
Q48: 10
Q49: 6
Q50: 1
Q51: 7
Q52: 1
Q53: 9
Q54: 9
Q55: 9
Q56: 9
Q57: 9
Q58: 9
Q59: 9
Q60: 9
Q61: 9
Q62: 9
Q63: 9
Q64: 5
Q65: 3
Q66: 7
Q67: 6
Q68: 1
Q69: 2
Q70: 8
Q71: 4
Q72: 7
Q73: 6
Q74: 5
Q75: 6
Q76: 1
Q77: 2
Q78: 1
Q79: 1
Q80: 1
Q81: 1
Q82: 9
Q83: 10
Q84: 10
Q85: 1
Q86: 9
Q87: 1
Q88: 1
Q89: 1
Q90: 1
Q91: 1
Q92: 10
Q93: 1
Q94: 1
Q95: 1
Q96: 1
Q97: 10
Q98: 10
Q99: 9
Q100: 3

เสร็จแล้ว! บันทึกที่: /content/drive/MyDrive/super-ai-engineer-season-6-rag-2569/data/sample_submission.csv


In [ ]:
# def rag_answer(row):
#     retrieved = retriever.invoke(row['question'])
#     context = "\n\n".join([doc.page_content for doc in retrieved]) ## ไม่ลองแบบ reverse แล้ว

#     choices_text = "\n".join([
#         f"ตัวเลือกที่ {i}: {row[f'choice_{i}']}"
#         for i in range(1, 11)
#     ])

#     messages = [
#         {
#             "role": "system",
#             "content": (
#                 "คุณเป็นผู้ช่วยตอบคำถามของร้านฟ้าใหม่ ร้านอิเล็กทรอนิกส์\n"
#                 "อ่านข้อมูลอ้างอิงทุก chunk ให้ครบและละเอียดก่อนตอบ\n"
#                 "ข้อมูลที่ต้องการอาจอยู่ใน chunk ใดก็ได้ ไม่จำเป็นต้องอยู่ chunk แรก\n\n"
#                 "กฎเหล็ก:\n"
#                 "1. ถ้าพบข้อมูลที่เกี่ยวข้องใน chunk ใดก็ตาม ให้ตอบตัวเลขของตัวเลือกที่ตรงที่สุดเสมอ\n"
#                 "2. ตอบ 9 เฉพาะเมื่ออ่านครบทุก chunk แล้วไม่มีข้อมูลที่เกี่ยวข้องจริงๆ เท่านั้น\n"
#                 "3. ตอบ 10 เฉพาะเมื่อคำถามไม่เกี่ยวกับร้านฟ้าใหม่หรือสินค้าในร้านเลย\n"
#                 "4. ห้ามตอบ 9 ถ้ายังมีข้อมูลใน chunk ที่เกี่ยวข้องกับคำถามแม้แต่น้อย\n"
#                 "5. ตอบเฉพาะตัวเลข 1-10 เท่านั้น ห้ามตอบอย่างอื่นเด็ดขาด"
#             )
#         },
#         {
#             "role": "user",
#             "content": (
#                 f"ข้อมูลอ้างอิง:\n{context}\n\n"
#                 f"คำถาม: {row['question']}\n\n"
#                 f"{choices_text}\n\n"
#                 "ตอบเฉพาะตัวเลข 1-10:"
#             )
#         }
#     ]

#     raw = ask_llm(messages, model="typhoon")

#     if raw:
#         match = re.search(r'\b([1-9]|10)\b', raw)
#         if match:
#             return int(match.group(1))
#     return 9  # default

# # วนทุกข้อ
# answers = []
# for _, row in questions_df.iterrows():
#     ans = rag_answer(row)
#     print(f"Q{row['id']}: {ans}")
#     answers.append({'id': int(row['id']), 'answer': ans})
#     pd.DataFrame(answers).to_csv(output_path, index=False)  # save ทุกข้อ

# print(f"\nเสร็จแล้ว! บันทึกที่: {output_path}")

In [ ]:
# for i in [3, 9, 21]:
#     row = questions_df.iloc[i]
#     retrieved = retriever.invoke(row['question'])
#     print(f"=== Q{row['id']}: {row['question']} ===")
#     for j, doc in enumerate(retrieved):
#         print(f"chunk {j+1}: {doc.page_content[:150]}")
#     print()

In [ ]:
# import pandas as pd
# df = pd.read_csv(output_path)
# print(df['answer'].value_counts().sort_index())

In [ ]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# from google.colab import userdata

# def download_embeddings():
#     """
#     Download and initialize Google Gemini Embeddings
#     """
#     # ดึง API Key จาก Colab Secrets (ต้องตั้งชื่อว่า GOOGLE_API_KEY หรือ Typhoon ตามที่คุณมี)
#     gemini_key = userdata.get('GOOGLE_API_KEY')

#     embeddings = GoogleGenerativeAIEmbeddings(
#         model=""models/gemini-embedding-2-preview"",
#         google_api_key=gemini_key
#     )
#     return embeddings

# # gemini_embedding = download_gemini_embeddings()
# # vector = gemini_embedding.embed_query('สวัสดีโลก')
# # print(len(vector)) # ปกติจะได้ 3072 มิติ